In [54]:
from langgraph.graph import StateGraph, START,END
from typing import TypedDict
from langchain_groq import ChatGroq
from dotenv import load_dotenv
load_dotenv()

True

In [55]:
class CrecketState(TypedDict):
    runs: int
    balls:int
    fours:int
    sixes:int
    strike_rate:float
    runs_from_boundaries:int
    balls_per_boundary:float

In [56]:
def calculate_strike_rate(state: CrecketState) -> CrecketState:
    run = state["runs"]
    ball = state["balls"]
    if ball > 0:
        strike_rate = (run / ball) * 100
    else:
        strike_rate = 0
    
    return {"strike_rate": strike_rate}

In [57]:
def calculate_runs_boundary(state: CrecketState) -> CrecketState:
    fours = state["fours"]
    sixes = state["sixes"]
    runs_from_fours = fours * 4
    runs_from_sixes = sixes * 6
    total_runs_from_boundaries = runs_from_fours + runs_from_sixes
    return {"runs_from_boundaries": total_runs_from_boundaries}

In [58]:
def balls_per_boundary(state: CrecketState) -> CrecketState:
    balls = state["balls"]
    total_boundaries = state["fours"] + state["sixes"]
    if total_boundaries > 0:
        balls_per_boundary = balls / total_boundaries
    else:
        balls_per_boundary = 0
    
    return {"balls_per_boundary": balls_per_boundary}

In [59]:
def summary(state: CrecketState) -> CrecketState:
    summary = f"Runs: {state['runs']}, Balls: {state['balls']}, Fours: {state['fours']}, Sixes: {state['sixes']}, Strike Rate: {state['strike_rate']:.2f}, Runs from Boundaries: {state['runs_from_boundaries']}, Balls per Boundary: {state['balls_per_boundary']:.2f}"
    return {"summary": summary}

In [60]:
graph = StateGraph(CrecketState)
graph.add_node("calculate_strike_rate", calculate_strike_rate)
graph.add_node("calculate_runs_boundary", calculate_runs_boundary)
graph.add_node("balls_per_boundary", balls_per_boundary)
graph.add_node("summary", summary)

graph.add_edge(START, "calculate_strike_rate")
graph.add_edge(START, "calculate_runs_boundary")
graph.add_edge(START, "balls_per_boundary")

graph.add_edge("calculate_strike_rate", "summary")
graph.add_edge("calculate_runs_boundary", "summary")
graph.add_edge("balls_per_boundary", "summary")

graph.add_edge("summary", END)

workflow = graph.compile()

In [61]:
initial_state = {
    "runs": 120,
    "balls": 80,
    "fours": 10,
    "sixes": 5,
}
final_state = workflow.invoke(initial_state)
print(final_state)

{'runs': 120, 'balls': 80, 'fours': 10, 'sixes': 5, 'strike_rate': 150.0, 'runs_from_boundaries': 70, 'balls_per_boundary': 5.333333333333333}
